# Remotion: бутылка и бриллианты

Собирает два MP4 в один ролик: сцена с бутылкой, затем сцена с бриллиантами. Итог: 1920×1080, 30 fps, H.264.

Загрузка выполняется прямо в ноутбук. Railway не используется.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
uploader = widgets.FileUpload(accept='.mp4', multiple=True, description='Выбрать два MP4')
display(uploader)
print('Выберите оба видеофайла. После выбора выполните следующую ячейку.')


In [ ]:
import os, pathlib, json, subprocess, shutil
work = pathlib.Path('/kaggle/working/remotion-case')
work.mkdir(parents=True, exist_ok=True)
items = uploader.value
if isinstance(items, dict):
    entries = list(items.items())
else:
    entries = [(x['name'], x) for x in items]
assert len(entries) >= 2, 'Нужно выбрать два MP4-файла.'
for name, item in entries:
    data = item['content']
    (work / pathlib.Path(name).name).write_bytes(bytes(data))
videos = sorted(work.glob('*.mp4'))
assert len(videos) >= 2, 'В папке должны быть два MP4.'
print('Первый файл (бутылка):', videos[0].name)
print('Второй файл (бриллианты):', videos[1].name)


In [ ]:
import subprocess, os
print(subprocess.check_output('node --version || true', shell=True, text=True))
if subprocess.call('command -v node >/dev/null 2>&1', shell=True) != 0:
    subprocess.run('apt-get update -qq && apt-get install -y -qq nodejs npm', shell=True, check=True)
subprocess.run(['npm','init','-y'], cwd=work, check=True, stdout=subprocess.DEVNULL)
subprocess.run(['npm','install','remotion@4.0.409','@remotion/cli@4.0.409','react@18.3.1','react-dom@18.3.1'], cwd=work, check=True)
subprocess.run(['npx','remotion','browser','ensure'], cwd=work, check=True)
print('Remotion установлен.')


In [ ]:
source = '''import React from 'react';\nimport {registerRoot} from 'remotion';\nimport {Composition, AbsoluteFill, OffthreadVideo, Sequence} from 'remotion';\nconst Clip = ({src}) => <AbsoluteFill><OffthreadVideo src={src} style={{width:'100%',height:'100%',objectFit:'cover'}} /></AbsoluteFill>;\nconst Main = ({bottle, diamonds}) => <AbsoluteFill style={{backgroundColor:'black'}}><Sequence from={0} durationInFrames={150}><Clip src={bottle}/></Sequence><Sequence from={150} durationInFrames={240}><Clip src={diamonds}/></Sequence></AbsoluteFill>;\nconst Root = () => <Composition id='BottleDiamondCase' component={Main} durationInFrames={390} fps={30} width={1920} height={1080} defaultProps={{bottle:'',diamonds:''}} />;\nregisterRoot(Root);\n'''
(work/'index.jsx').write_text(source)
videos = sorted(work.glob('*.mp4'))
props = {'bottle': videos[0].resolve().as_uri(), 'diamonds': videos[1].resolve().as_uri()}
(work/'props.json').write_text(json.dumps(props))
print('Сцены подготовлены: 5 секунд + 8 секунд.')


In [ ]:
output = pathlib.Path('/kaggle/working/bottle-diamond-case.mp4')
cmd = ['npx','remotion','render','index.jsx','BottleDiamondCase',str(output),'--props',str(work/'props.json'),'--codec','h264','--concurrency','2']
subprocess.run(cmd, cwd=work, check=True)
print('Готово:', output, 'Размер:', round(output.stat().st_size/1024/1024,1), 'МБ')


In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/bottle-diamond-case.mp4'))
